# Water Quality Prediction & Evolutionary Optimisation
**Date:** April 2025  

This notebook covers two parts:
- **Part 1:** Predicting nitrite levels in water samples using machine learning regression models (Linear Regression, Random Forest, Neural Network)
- **Part 2:** Optimising the McCormick function using a Hill Climber and Evolutionary Algorithm

---

## Table of Contents
1. [Libraries & Setup](#1-libraries--setup)
2. [Part 1 – Machine Learning](#1)
   - [2.1 Load Data](#1)
   - [2.2 Exploratory Data Analysis](#2.2)

<a id="1-libraries--setup"></a>
---
## 1. Libraries & Setup

In [ ]:
# core libraries
import numpy as np
import pandas as pd

#for visualization
import matplotlib.pyplot as plt
import seaborn as sns

#Regression models
import sklearn.linear_model as linearRegression
import sklearn.ensemble as RandomForestRegressor
import sklearn.neural_network as MLPRegressor

#Metrices 
from sklearn.metrics import mean_squared_error, r2_score

# Plot styling
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

print('All libraries loaded successfully.')

<a id="1"></a>
---
## Part 1 – Machine Learning

### 2.1 Load Data

In [ ]:
#load the data using pandas
data_file = pd.read_csv("../data/e1_nutrients.csv")

Used head(), info() and describe() to explore the data and understand the charactiristic of the data.

In [ ]:
data_file.head()        # first 5 rows
data_file.info()        # column names, data types, missing values
data_file.describe()    # min, max, mean, std for every column

<a id="2.2"></a>
---
### 2.2 Exploratory Data Analysis

In [ ]:
# Visualize feature distributions using KDE plots
fig, axes = plt.subplots(2, 3, figsize=(14, 10))
# Flatten the axes array for easy iteration
for ax, col in zip(axes.flatten(), data_file.columns):
    # Create a KDE plot for each feature
    sns.kdeplot(data_file[col], ax=ax, fill=True, color='steelblue')
    ax.set_title(col)

# Adjust layout and show the plot
plt.suptitle('Feature Distributions (KDE)', fontsize=16)
plt.tight_layout()
plt.show()

## Distribution Charts Observation
>Previously, I chose a histogram plot to visualise the data, which, compared to KDE, was missing some critical information.
### The KDE reveals:
1. DEPTH The KDE shows 6 distinct peaks, one for each depth value. This actually visually proves it's discrete rather than continuous, far more clearly than the histogram did.
2. NITRATE+NITRITE has a clear bimodal distribution (two humps — one near 0 and a second around 5-6). This is a significant finding. It suggests two distinct water conditions exist in the dataset, possibly seasonal or driven by depth.
3. SILICATE also shows a bimodal shape with peaks around 0.5 and 2. Again suggests two different water states.
4. PHOSPHATE similarly shows a small secondary peak around 1.5 after the main peak near 0.2.

>The KDE plots reveal that Depth takes exactly 6 discrete values (0, 10, 20, 30, 
40 and 60 metres), visible as 6 distinct peaks, confirming a controlled repeated 
sampling design. NITRITE and AMMONIA are heavily right-skewed with a sharp peak 
near zero and a long tail, indicating most samples have low concentrations, with 
Sometimes, extreme readings which would require outlier handling after being confirmed by other visualisation methods. More interestingly, 
NITRATE+NITRITE, SILICATE and PHOSPHATE all display bimodal distributions, two 
distinct peaks. This suggests the dataset captures two different water conditions, 
possibly caused by the seasonal variation or possible depth-related nutrient stratification. 
These patterns suggest the relationships in this data are non-linear and complex.


In [ ]:
# Correlation heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(data_file.corr(), annot=True, fmt='.2f', cmap='coolwarm', square=True)
plt.title('Correlation Matrix')
plt.tight_layout()
plt.savefig('../Analytics/correlation_heatmap.png', dpi=150)
plt.show()

## Correlation Heatmap Observation
What does the heatmap tell me?
1. Looking at the NITRATE+NITRITE and PHOSPHATE numbers(0.80), we can conclude that their relationship is the strongest. When one side is high, we can assume the other side could also be high.
2. NITRATE+NITRITE and SILICATE (0.69) have a strong relationship, and again, when one increases, you can see the increase on the other as well.
3. SILICATE and PHOSPHATE (0.61) Moderate positive relationship, same pattern.
4. AMMONIA and NITRATE+NITRITE (-0.22) Slight negative relationship, slightly unusual.

### Most important
### NITRITE and everything else (0.03 to 0.31)
Our most important nutrient, which is the target variable (for this prediction model), has a low relationship with everything else.

### What does that mean?
>This means predicting nitrite is genuinely difficult, and linear relationships alone won't capture it well. The weak correlations between NITRITE and all features suggest that nitrite behaves somewhat independently from the other nutrients, which means Linear Regression will likely struggle compared to Random Forest or Neural Network, which can capture non-linear relationships. 
#### *** These are predictions I made before even creating the model, solely based on the correlation heatmap. *** #### 
 


In [ ]:
# Box plots to identify outliers
data_file.plot(kind='box', subplots=True, layout=(3, 4), figsize=(16, 10), sharex=False, sharey=False)
plt.suptitle('Box Plots – Outlier Detection', fontsize=16)
plt.tight_layout()
plt.savefig('../Analytics/boxplots.png', dpi=150)
plt.show()

## Box Plot Observation
What does it tell me?
1. DEPTH No outliers, clean even spread across 0-60. Confirms what we already knew: controlled sampling.
2. NITRITE Massive amount of outliers shown as black dots above the whisker. The box itself is tiny and squashed near 0, meaning that 50% of all the values are extremely low, but there are hundreds of readings that are unusually high. This confirms the heavy right skew and tells me outlier handling is definitely needed here.
3. NITRATE+NITRITE Large spread in the box, no extreme outliers shown. More variation in the normal range than NITRITE.
4. AMMONIA shows simmilar behaviour as NITRITE.The box is small and there is a huge dense cluster of outliers above.That means there are hundereds of AMMONIA readings that go well above the normal range.
5. SILICATE Only a handful of outliers (the small circles), and the box has a reasonable spread. Much cleaner than NITRITE or AMMONIA. However the density of the outliers in the 5.8 area is unclear. 
6. PHOSPHATE A few scattered outliers above the whisker but nothing as extreme as NITRITE or AMMONIA. Relatively clean however again the density of the outliers in the narow section of approximitly 1.3 is not very clear. 

### What does that mean for me?

>NITRITE and AMMONIA have the most severe outlier problems, hundreds of data points sitting well above the normal range. These need to be handled in the data preparation step with the IQR method before training the models, otherwise those extreme values will disproportionately influence how the models learn and they skew the results.


